In [1]:
import numpy as np
import pandas as pd

In [5]:
votes = pd.read_csv("https://voteview.com/static/data/out/votes/HS119_votes.csv")

In [6]:
votes.head(3).T

,0,1,2
congress,119,119,119
chamber,House,House,House
rollnumber,1,1,1
icpsr,14854,14863,14873
cast_code,1,1,6
prob,99.6,78.9,100.0


## 🐼 Skill: filter columns

In [7]:
votes = votes[['chamber', 'rollnumber', 'icpsr', 'cast_code']] ## keep columns

## or do this: 
## votes = votes.drop(['congress', 'prob'], axis = 1) ## drop columns

We need to do a many to many kind of cross join

example below:


In [10]:
data1 = [{'id': 1, 'x': 'a'}, {'id': 1, 'x': 'b'}, {'id': 1, 'x': 'c'}]
data2 = [{'id': 1, 'y': 'd'}, {'id': 1, 'y': 'e'}, {'id': 1, 'y': 'f'}]
#pd.DataFrame(data1)
#pd.DataFrame(data2)

mergedata = pd.merge(pd.DataFrame(data1), pd.DataFrame(data2), on = 'id')

mergedata

,id,x,y
0,1,a,d
1,1,a,e
2,1,a,f
3,1,b,d
4,1,b,e
5,1,b,f
6,1,c,d
7,1,c,e
8,1,c,f


## 🐼 skill: many-to-many joins (usually we don't want this! but here we do)

In [11]:
vote_compare = pd.merge(votes, votes,
                        on = ['chamber', 'rollnumber'])

In [12]:
vote_compare

,chamber,rollnumber,icpsr_x,cast_code_x,icpsr_y,cast_code_y
0,House,1,14854,1,14854,1
1,House,1,14854,1,14863,1
2,House,1,14854,1,14873,6
3,House,1,14854,1,15029,6
4,House,1,14854,1,15433,6
...,...,...,...,...,...,...
135162640,Senate,915,49703,1,42506,1
135162641,Senate,915,49703,1,42507,1
135162642,Senate,915,49703,1,42508,1
135162643,Senate,915,49703,1,49308,6


## 🐼 skill: more dropping

In [13]:
vote_compare = vote_compare.drop(['chamber', 'rollnumber'], axis=1)

In [14]:
vote_compare

,icpsr_x,cast_code_x,icpsr_y,cast_code_y
0,14854,1,14854,1
1,14854,1,14863,1
2,14854,1,14873,6
3,14854,1,15029,6
4,14854,1,15433,6
...,...,...,...,...
135162640,49703,1,42506,1
135162641,49703,1,42507,1
135162642,49703,1,42508,1
135162643,49703,1,49308,6


## 🐼 skill: create a new column

In [15]:
vote_compare['agree'] = vote_compare['cast_code_x'] == vote_compare['cast_code_y']
vote_compare

,icpsr_x,cast_code_x,icpsr_y,cast_code_y,agree
0,14854,1,14854,1,True
1,14854,1,14863,1,True
2,14854,1,14873,6,False
3,14854,1,15029,6,False
4,14854,1,15433,6,False
...,...,...,...,...,...
135162640,49703,1,42506,1,True
135162641,49703,1,42507,1,True
135162642,49703,1,42508,1,True
135162643,49703,1,49308,6,False


In [16]:
vote_compare = vote_compare.drop(['cast_code_x', 'cast_code_y'], axis=1)

## 🐼 skill: filter rows

In [17]:
vote_compare = vote_compare.query('icpsr_x != icpsr_y')

In [18]:
vote_compare

,icpsr_x,icpsr_y,agree
1,14854,14863,True
2,14854,14873,False
3,14854,15029,False
4,14854,15433,False
5,14854,15448,False
...,...,...,...
135162639,49703,42505,True
135162640,49703,42506,True
135162641,49703,42507,True
135162642,49703,42508,True


## 🐼 skill: rename columns

In [19]:
vote_compare = vote_compare.rename({'icpsr_x': 'icpsr',
                                    'icpsr_y': 'comparison_member'}, axis = 1)

In [20]:
vote_compare

,icpsr,comparison_member,agree
1,14854,14863,True
2,14854,14873,False
3,14854,15029,False
4,14854,15433,False
5,14854,15448,False
...,...,...,...
135162639,49703,42505,True
135162640,49703,42506,True
135162641,49703,42507,True
135162642,49703,42508,True


## 🐼 skill: sorting

In [21]:
vote_compare = vote_compare.sort_values(['icpsr', 'comparison_member'], ascending=True)

In [22]:
vote_compare

,icpsr,comparison_member,agree
125985887,14226,14435,False
125995688,14226,14435,False
126005292,14226,14435,False
126015093,14226,14435,False
126024894,14226,14435,False
...,...,...,...
37045188,99912,91980,True
37232677,99912,91980,True
37420166,99912,91980,True
37794279,99912,91980,True


## 🐼 skill: data aggregation

In [24]:
vote_compare = vote_compare.groupby(['icpsr', 'comparison_member']).agg({'agree': 'mean'}).reset_index()
vote_compare

,icpsr,comparison_member,agree
0,14226,14435,0.028415
1,14226,14858,0.077596
2,14226,14871,0.060109
3,14226,14921,0.798907
4,14226,15021,0.123497
...,...,...,...
215349,99912,42505,1.000000
215350,99912,42506,1.000000
215351,99912,49308,0.000000
215352,99912,49703,0.972414


In [25]:
## user supplies 14226, for example
user = '14226'
vote_compare.query(f'icpsr == {user}').sort_values('agree', ascending=False).head(10)

,icpsr,comparison_member,agree
103,14226,99912,1.000000
91,14226,42304,1.000000
57,14226,41102,1.000000
20,14226,21166,0.961749
89,14226,42302,0.958470
49,14226,40707,0.958470
59,14226,41107,0.957377
72,14226,41505,0.957377
94,14226,42502,0.955191
65,14226,41302,0.955191


In [26]:
vote_compare.query(f'icpsr == {user}').sort_values('agree', ascending=True).head(10)

,icpsr,comparison_member,agree
0,14226,14435,0.028415
64,14226,41301,0.030601
37,14226,29147,0.031694
10,14226,20707,0.037158
8,14226,20330,0.046995
52,14226,40908,0.048087
2,14226,14871,0.060109
101,14226,49308,0.061202
29,14226,21706,0.062295
76,14226,41706,0.063388


In [28]:
## How this will work if we grab the contrans class from canvas
import os
os.chdir('..')
from contrans import contrans

ct = contrans()

In [32]:
ct.get_vote_similarity_data()

In [33]:
vote_compare = pd.read_parquet('data/raw/vote_compare.parquet')
vote_compare

,bioname,comparison_member,agree
0,"GRASSLEY, Charles Ernest","MARKEY, Edward John",0.028415
1,"GRASSLEY, Charles Ernest","SCHUMER, Charles Ellis (Chuck)",0.077596
2,"GRASSLEY, Charles Ernest","WYDEN, Ronald Lee",0.060109
3,"GRASSLEY, Charles Ernest","MCCONNELL, Addison Mitchell (Mitch)",0.798907
4,"GRASSLEY, Charles Ernest","DURBIN, Richard Joseph",0.123497
...,...,...,...
215349,"TRUMP, Donald John","HUSTED, Jon Allen",1.000000
215350,"TRUMP, Donald John","MOODY, Ashley Brooke",1.000000
215351,"TRUMP, Donald John","MURRAY, Patty",0.000000
215352,"TRUMP, Donald John","COLLINS, Susan Margaret",0.972414
